## COSC 524: Natural Lanuage Processing Final Project
### Author: Jacob Mendez

#### Introduction

For my chatbot project, my project is for the card game Magic: The Gathering. For the project, I am implementing a Seq2seq model, with a corpus of training documents, for the purpose of building a chatbot to assist with rulings for the game. In Magic, there are over 900 numbered rules, with a great portion also containing subrules. These rules are also not static, and must be updated as new cards introduce new mechanics and work seamlessly with pre established cards and rules. The goal is to build a chatbot that can quickly reference a rule or game mechanic and provide context for the rule itself.

#### Documents:

##### Rules

In [ ]:
with open("MTG_Rules.txt", 'r', encoding='utf-8') as file:
        text = file.read()

print(text)

##### Cards

In [ ]:
with open("cards.txt", 'r', encoding='utf-8') as file:
        text = file.read()

print(text)

##### Decks

In [ ]:
with open("precon_decks.txt", 'r', encoding='utf-8') as file:
        text = file.read()

print(text)

## Chatbot

### Imports and data

In [ ]:
import json
from datasets import Dataset
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments

with open("dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

inputs = [item["input_text"] for item in data]
targets = [item["target_text"] for item in data]

dataset = Dataset.from_dict({
    "input_text": inputs,
    "target_text": targets
})

/home/jacob/CodeSpace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1778063966.580411  991027 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778063967.327551  991027 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778063967.240809  991027 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-poin

#### Separating Text

Text cleaning was performed to unify how all training data was fed in. Text was put to lowercase, had all punctuation removed, and provided a start and end token.

In [2]:
inputs = [item["input_text"] for item in data]
targets = [item["target_text"] for item in data]

dataset = Dataset.from_dict({
    "input_text": inputs,
    "target_text": targets 
})

### Train/Test Split

In [3]:
dataset = dataset.train_test_split(test_size=0.2)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

### Tokenization

Input and target texts from data are tokenized and converted to a list of tokens. Sizes are taken for embeddings, and the longest is found to ensure all sequences can be padded such that they are all this length.

In [4]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

def tokenize(sample):
    model_inputs = tokenizer(
        sample["input_text"],
        max_length = 128,
        truncation = True,
        padding= "max_length"
    )

    labels = tokenizer(
        sample["target_text"],
        max_length= 128,
        truncation = True,
        padding= "max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 2000/2000 [00:00<00:00, 4573.87 examples/s]


In [9]:
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


### Training Configuration

In [7]:
train_args = TrainingArguments(
    output_dir="./bart-mtg",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_steps=100,
    save_total_limit=2,
    fp16=True
)

### Model

In [10]:
trainer = Trainer(
    model= model,
    args = train_args,
    train_dataset= train_dataset,
    eval_dataset= test_dataset,
    tokenizer= tokenizer
)


/tmp/ipykernel_991027/87652715.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Training

In [13]:
trainer.train()
trainer.save_model("./bart-mtg-model")
tokenizer.save_pretrained("./bart-mtg-model")

Step,Training Loss
100,0.416900
200,0.397100
300,0.390800
400,0.382700
500,0.382100
600,0.361100
700,0.363500
800,0.370800
900,0.338900
1000,0.357700


/home/jacob/CodeSpace/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


('./bart-mtg-model/tokenizer_config.json',
 './bart-mtg-model/special_tokens_map.json',
 './bart-mtg-model/vocab.json',
 './bart-mtg-model/merges.txt',
 './bart-mtg-model/added_tokens.json')

In [14]:
tokenizer = BartTokenizer.from_pretrained("./bart-mtg-model")
model = BartForConditionalGeneration.from_pretrained("./bart-mtg-model")

In [15]:
def chatbot(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=64,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0])


In [17]:
while True:
    user_input = input("You")
    if user_input.lower() in ["quit", "exit", "bye", "goodbye", "stop"]:
        break
    print("Bot: ", chatbot(user_input))

Bot:  </s><s>Forest — Basic Land — Forest. Effect: ({T}: Add {G}.)</s>
Bot:  </s><s>Tidus deck — Land. Effect: This land enters tapped. When this land enters, return a land you control to its owner's hand. {T}: Add {U} or {B}.</s>
Bot:  </s><s>Tidus — Creature — Insect. Effect: When this creature enters, you may search your library for a basic land card, put it onto the battlefield tapped, then shuffle.</s>
